# REFLECT – RAG Answer Quality Eval
Retrieves top-k chunks per query, generates a grounded answer, then scores faithfulness, relevancy, and context precision.

## Config

In [6]:
from pathlib import Path
import json
from statistics import mean, pstdev
from typing import Any
import pandas as pd
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

SCRIPT_DIR      = Path(".").resolve()
EVAL_DATA_PATH  = SCRIPT_DIR / "eval_data"
RESULTS_PATH    = SCRIPT_DIR / "eval_results"
OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL     = "nomic-embed-text"

# ⚠️  'gpt-oss:20b' is a placeholder – replace with your actual Ollama tag.
# ⚠️  Avoid using the same models as both generator AND judge (self-grading bias).
MODELS_TO_TEST  = ["mistral", "llama3", "gpt-oss:20b"]
JUDGE_MODELS    = ["mistral", "llama3"]

TOP_K           = 5
MAX_QUERIES     = None   # e.g. 3 for a quick smoke test
REQUEST_TIMEOUT = 600.0

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print("Config OK")


Config OK


## Prompts

In [7]:
ANSWER_PROMPT = """You are an evidence-grounded reflection assistant.

Answer the user's question using only the retrieved journal context.
If context is insufficient, say what is missing instead of inventing details.

Question:
{query}

Retrieved context:
{context}

Answer:"""

JUDGE_PROMPT = """You are grading a RAG answer. Be strict and unsparing.
Respond only with valid JSON, no extra text.

Question:
{query}

Retrieved context:
{context}

Model answer:
{answer}

Score each 1-5:
- faithfulness: every claim is supported by retrieved context; penalize invented details hard.
- answer_relevancy: directly answers the question without vague filler.
- context_precision: uses relevant context points, not generic statements disconnected from evidence.

{{"faithfulness": N, "answer_relevancy": N, "context_precision": N, "reason": "one sentence"}}"""

TEST_QUERIES = [
    "What recurring patterns appear in how I handle work stress?",
    "What does my relationship with productivity look like?",
    "How do I talk about my relationships with other people?",
    "What themes come up when I am having a low energy day?",
    "What am I avoiding or procrastinating on?",
    "Where do I sound conflicted between values and actions?",
    "What tradeoffs keep appearing in my decisions?",
]
print("Prompts loaded")


Prompts loaded


## Helpers

In [8]:
def build_llm(model_name: str) -> Ollama:
    extra = {"think": False} if "qwen" in model_name.lower() else {}
    return Ollama(model=model_name, base_url=OLLAMA_BASE_URL,
                  request_timeout=REQUEST_TIMEOUT, additional_kwargs=extra)

def extract_json_dict(raw: str) -> dict:
    text = raw.strip().lstrip("```").lstrip("json").strip().rstrip("```").strip()
    try:
        p = json.loads(text); return p if isinstance(p, dict) else {}
    except json.JSONDecodeError:
        s, e = text.find("{"), text.rfind("}") + 1
        if s == -1 or e <= s: return {}
        try: p = json.loads(text[s:e]); return p if isinstance(p, dict) else {}
        except: return {}

def to_score(v) -> int | None:
    try: s = int(v)
    except: return None
    return max(1, min(5, s))

def generate_answer(llm, query: str, context: str) -> str:
    return str(llm.complete(ANSWER_PROMPT.format(query=query, context=context))).strip()

def judge_answer(judge_llm, query: str, context: str, answer: str) -> dict:
    raw = str(judge_llm.complete(JUDGE_PROMPT.format(
        query=query, context=context, answer=answer))).strip()
    return extract_json_dict(raw)

def aggregate_judgements(payloads: dict) -> dict:
    metrics = ("faithfulness", "answer_relevancy", "context_precision")
    per = {}; reasons = []; valid = 0
    for jname, payload in payloads.items():
        row = {m: to_score(payload.get(m)) for m in metrics}
        per[jname] = row
        if any(v is not None for v in row.values()): valid += 1
        r = payload.get("reason","")
        if isinstance(r, str) and r.strip(): reasons.append(f"{jname}: {r.strip()}")
    agg = {}
    for m in metrics:
        vals = [row[m] for row in per.values() if row[m] is not None]
        agg[m] = round(mean(vals), 3) if vals else None
        agg[f"{m}_judge_std"] = (None if not vals else 0.0 if len(vals)==1
                                 else round(pstdev(vals), 3))
    n = max(1, len(payloads))
    agg["judge_valid_count"] = valid
    agg["judge_valid_rate"]  = round(valid / n, 3)
    agg["judge_reasons"]     = " | ".join(reasons)
    agg["judge_scores_json"] = json.dumps(per, ensure_ascii=True)
    return agg

print("Helpers loaded")


Helpers loaded


## Build Index
> Tip: persist the index to disk with `StorageContext` to avoid re-embedding on every run.

In [9]:
documents = SimpleDirectoryReader(str(EVAL_DATA_PATH)).load_data()
print(f"Loaded {len(documents)} document(s)")

embed = OllamaEmbedding(model_name=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
index = VectorStoreIndex.from_documents(documents, embed_model=embed)
print("Index built")


Loaded 15 document(s)
Index built


## Run Eval

In [ ]:
queries    = TEST_QUERIES[:MAX_QUERIES] if MAX_QUERIES else TEST_QUERIES
judge_llms = {j: build_llm(j) for j in JUDGE_MODELS}

print(f"Queries: {len(queries)}  |  Generators: {len(MODELS_TO_TEST)}  |  Judges: {list(judge_llms)}")
rows = []

for model_name in MODELS_TO_TEST:
    print(f"\n── {model_name} ──")
    gen = build_llm(model_name)
    for query in queries:
        retriever = index.as_retriever(similarity_top_k=TOP_K)
        nodes     = retriever.retrieve(query)
        chunks    = [n.get_content() for n in nodes]
        context   = "\n\n---\n\n".join(chunks)
        answer    = generate_answer(gen, query, context)
        payloads  = {j: judge_answer(jllm, query, context, answer)
                     for j, jllm in judge_llms.items()}
        agg = aggregate_judgements(payloads)
        print(f"  Q: {query[:65]}...")
        print(f"  A: {answer[:100]}{'...' if len(answer)>100 else ''}")
        rows.append({"model": model_name, "query": query,
                     "retrieved_chunks": len(chunks),
                     "answer": answer, **agg})

df = pd.DataFrame(rows)
print("\nDone.")


Queries: 7  |  Generators: 3  |  Judges: ['mistral', 'llama3']

── mistral ──


: 

## Summary

In [ ]:
summary = (
    df.groupby("model")
    .agg(samples=("answer","count"),
         faithfulness=("faithfulness","mean"),
         answer_relevancy=("answer_relevancy","mean"),
         context_precision=("context_precision","mean"),
         judge_valid_rate=("judge_valid_rate","mean"))
    .round(3)
    .sort_values(["faithfulness","answer_relevancy","context_precision"], ascending=False)
)

per_path = RESULTS_PATH / "rag_eval.csv"
sum_path = RESULTS_PATH / "rag_eval_summary.csv"
df.to_csv(per_path, index=False)
summary.to_csv(sum_path)

print(summary)
print(f"\nSaved: {per_path}\n       {sum_path}")
